# Feature Engineering

## Objective
Construct domain-specific features that encode electrical engineering knowledge:
1. **Power quality indicators** — PF deviation, voltage sag/swell, frequency excursion
2. **Interaction features** — Cross-variable products and ratios
3. **Rolling statistics** — Windowed mean, std, skewness for temporal patterns
4. **Rate-of-change** — First/second derivatives for transient detection
5. **Statistical features** — Z-scores for outlier sensitivity

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.abspath('..'))

from src.feature_engineering import (
    compute_power_quality_features,
    compute_interaction_features,
    compute_rolling_statistics,
    compute_rate_of_change,
    compute_statistical_features,
    feature_engineering_pipeline,
)

DATA_DIR = os.path.join('..', 'data')
print('Libraries loaded')

In [ ]:
plant_df = pd.read_csv(os.path.join(DATA_DIR, 'power_plant_synthetic.csv'))
uci_df = pd.read_csv(os.path.join(DATA_DIR, 'uci_grid_processed.csv'))
print(f'Plant: {plant_df.shape}, UCI: {uci_df.shape}')

## 3.1 Power Quality Domain Features

These features encode expert knowledge about power system failure modes:
- **PF deviation** from unity indicates reactive power issues
- **Voltage deviation** from nominal (11 kV) indicates supply problems
- **Frequency deviation** from 50 Hz indicates load-generation imbalance
- **Thermal efficiency** proxy detects combustion/cooling anomalies

In [ ]:
plant_df, pq_features = compute_power_quality_features(plant_df)
print(f'\nPower quality features: {pq_features}')
plant_df[pq_features].describe().round(4)

## 3.2 Interaction Features

Cross-variable interactions capture failure modes invisible in individual features. For example, simultaneous voltage drop + current spike indicates a short-circuit fault.

In [ ]:
plant_df, interact_features = compute_interaction_features(plant_df)
print(f'\nInteraction features: {interact_features}')

## 3.3 Rolling Statistics

Capture temporal patterns using sliding windows. Rolling volatility (std) is particularly diagnostic — sudden spikes indicate transient events.

In [ ]:
rolling_cols = ['voltage_kv', 'current_a', 'load_mw', 'exhaust_temp_c', 'frequency_hz']
plant_df, roll_features = compute_rolling_statistics(plant_df, rolling_cols, windows=[5, 10])

## 3.4 Rate of Change Features

First and second derivatives detect abrupt transitions — critical for identifying the onset of faults.

In [ ]:
plant_df, roc_features = compute_rate_of_change(plant_df, rolling_cols)

## 3.5 Feature Engineering Summary

In [ ]:
all_engineered = pq_features + interact_features + roll_features + roc_features
original_features = [c for c in plant_df.columns if c not in all_engineered + ['anomaly', 'fault_type', 'source']]

print(f'Original features:    {len(original_features)}')
print(f'Power quality:        +{len(pq_features)}')
print(f'Interactions:         +{len(interact_features)}')
print(f'Rolling statistics:   +{len(roll_features)}')
print(f'Rate of change:       +{len(roc_features)}')
print(f'---')
print(f'Total features:       {len(original_features) + len(all_engineered)}')

## 3.6 Feature Importance Preview

Use a quick Random Forest to rank the engineered features before model training.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

all_feature_cols = original_features + all_engineered
all_feature_cols = [c for c in all_feature_cols if c in plant_df.columns]

# Drop any rows with NaN from rolling/diff operations
plant_clean = plant_df[all_feature_cols + ['anomaly']].dropna()

rf_quick = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
rf_quick.fit(plant_clean[all_feature_cols], plant_clean['anomaly'])

importances = pd.Series(rf_quick.feature_importances_, index=all_feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
importances.head(25).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Feature Importance')
ax.set_title('Top 25 Features (Quick RF Ranking)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/figures/feature_importance_preview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.7 Save Engineered Data

In [ ]:
plant_df.to_csv(os.path.join(DATA_DIR, 'plant_engineered.csv'), index=False)

import json
with open(os.path.join(DATA_DIR, 'engineered_feature_names.json'), 'w') as f:
    json.dump({'original': original_features, 'engineered': all_engineered, 'all': all_feature_cols}, f, indent=2)

print(f'Saved engineered dataset: {plant_df.shape}')
print(f'Saved feature names to data/engineered_feature_names.json')

## Summary

Feature engineering expanded the power plant dataset from 16 to 90+ features through domain-specific transformations. Key engineered features include power factor deviation, thermal efficiency, and rolling volatility measures.
